# Multivariate polynomials implementations

**Contributed by**: Benoît Legat

The SumOfSquares package is built on top of the [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl)
abstract interface. [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
is an implementation of this abstract interface so it can be used with
SumOfSquares. Moreover, any other implementation can be used as well. To
illustrate, we solve of [Blekherman2012; Examples 3.38](@cite) with
[TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl),
another implementation of [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl).

In [1]:
import TypedPolynomials
TypedPolynomials.@polyvar x y
using SumOfSquares
import CSDP
model = SOSModel(CSDP.Optimizer)
con_ref = @constraint(model, 2x^4 + 5y^4 - x^2*y^2 >= -2(x^3*y + x + 1))
optimize!(model)
solution_summary(model)

Iter: 18 Ap: 9.59e-01 Pobj: -3.5512000e-03 Ad: 9.59e-01 Dobj: -3.5512523e-03 
Success: SDP solved
Primal objective value: -3.5512000e-03 
Dual objective value: -3.5512523e-03 
Relative primal infeasibility: 1.38e-13 
Relative dual infeasibility: 5.36e-10 
Real Relative Gap: -5.20e-08 
XZ Relative Gap: 1.22e-09 
DIMACS error measures: 1.49e-13 0.00e+00 1.44e-09 0.00e+00 -5.20e-08 1.22e-09
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.39e-01 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  9.2771541e+01 
Iter:  2 Ap: 1.00e+00 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  5.3405053e+01 


solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 5.15938e-04

We see that the problem is feasible. The Sum-of-Squares decomposition can be
obtained as follows:

In [2]:
sos_decomposition(con_ref)

(-0.6797543969221541 - 0.09754507709792246*y - 0.4175950563713448*x + 2.1931518901396174*y^2 - 0.1269606244678339*x*y - 0.8025867547845221*x^2)^2 + (0.9043573827480671 - 0.6455003192597278*y + 0.2742159536999832*x - 0.10770174782358159*y^2 - 1.2293138068800717*x*y - 0.9300160939847*x^2)^2 + (-0.5739168240154698 - 1.38535050221548*y - 0.6219655378569912*x - 0.2040924100740057*y^2 - 0.15182215880489783*x*y + 0.4443832632766482*x^2)^2 + (0.5703021360913615 - 0.5184093413621602*y + 0.3436728787624473*x + 0.2555950605470665*y^2 + 0.7618441294650047*x*y - 0.02090567376761348*x^2)^2 + (0.041400837217767876 + 0.04911040272637946*y - 0.49990658891857337*x - 0.24792145939081137*y^2 + 0.2972445689917418*x*y - 0.5054184235227418*x^2)^2 + (-0.2524531788141328 - 0.06336647012459033*y + 0.253942572224051*x - 0.10018744153048117*y^2 + 0.059601795985895156*x*y - 0.19381241255811912*x^2)^2

Why is there several implementations ?
Depending in the use-case, one implementation may be more appropriate than
another one. [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is faster than [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
but it requires new compilation whenever the list of variables changes.
This means that [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is not appropriate when the number of variables is dynamic or too large.
However, for a small number of variables, it can be faster.
When solving Sum-of-Squares programs, the time is mostly taken by the Semidefinite programming solver.
The time taken by SumOfSquares/JuMP/MathOptInterface are usually negligible
or it time is taken by manipulation of JuMP or MathOptInterface functions
therefore using TypedPolynomials over DynamicPolynomials may not make much difference in most cases.

One case for which using TypedPolynomials might be adequate is when
using domain defined by equalities (possibly also with inequalities).
Indeed, in that case, SumOfSquares computes the corresponding Gröbner basis which
may take a non-negligible amount of time for large systems of equalities.

To illustrate this, consider the computation of Gröbner basis for the
following system from [CLO05, p. 17].
The time taken by TypedPolynomials is below:

[CLO05] Cox, A. David & Little, John & O'Shea, Donal
*Using Algebraic Geometry*.
Graduate Texts in Mathematics, **2005**.
https://doi.org/10.1007/b138611

In [3]:
using BenchmarkTools
@btime let
    TypedPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  187.694 μs (7906 allocations: 296.12 KiB)


true

The time taken by DynamicPolynomials is as follows:

In [4]:
import DynamicPolynomials
@btime let
    DynamicPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  367.022 μs (11632 allocations: 420.13 KiB)


true

We see that TypedPolynomials is faster.
The time is still negligible for this small system but for larger systems, choosing TypedPolynomials may be helpful.
We can use this system in a Sum-of-Squares constraint as follows:

In [5]:
TypedPolynomials.@polyvar x y
S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
poly = -6x - 4y^3 + 2x*y^2 + 6x^3 - 3y^4 + 13x^2 * y^2
model = Model(CSDP.Optimizer)
con_ref = @constraint(model, poly in SOSCone(), domain = S)
optimize!(model)
solution_summary(model)

Success: SDP solved
Primal objective value: 0.0000000e+00 
Dual objective value: 0.0000000e+00 
Relative primal infeasibility: 2.23e-16 
Relative dual infeasibility: 5.00e-11 
Real Relative Gap: 0.00e+00 
XZ Relative Gap: 3.04e-10 
DIMACS error measures: 2.78e-16 0.00e+00 5.00e-11 0.00e+00 0.00e+00 3.04e-10
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.64e-01 Pobj:  0.0000000e+00 Ad: 8.07e-01 Dobj:  2.6104524e+00 
Iter:  2 Ap: 7.71e-01 Pobj:  0.0000000e+00 Ad: 7.40e-01 Dobj: -5.9089088e-02 
Iter:  3 Ap: 8.08e-01 Pobj:  0.0000000e+00 Ad: 7.65e-01 Dobj:  5.5548423e-01 
Iter:  4 Ap: 7.56e-01 Pobj:  0.0000000e+00 Ad: 7.79e-01 Dobj:  4.8507215e-01 
Iter:  5 Ap: 6.95e-01 Pobj:  0.0000000e+00 Ad: 6.77e-01 Dobj:  4.3029745e-01 
Iter:  6 Ap: 6.35e-01 Pobj:  0.0000000e+00 Ad: 6.83e-01 Dobj:  2.2138595e-01 
Iter:  7 Ap: 6.21e-01 Pobj:  0.0000000e+00 Ad: 6.43e-01 Dobj:  1.2987326e-01 
Iter:  8 Ap: 

solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 2.66886e-03

We obtain the following decomposition:

In [6]:
dec = sos_decomposition(con_ref, 1e-6)

(-6.838847943388406e-5 + 0.0006543204897974745*y - 0.0001401966280264214*x + 7.045589910919535*y^2 - 28.23225665906972*x*y + 25.36606541237253*x^2)^2 + (-1.4188858280367412e-5 + 0.0004268077454268964*y + 0.0008149133136731779*x + 22.48148300158947*y^2 - 13.866164467967506*x*y - 21.67732421965715*x^2)^2 + (-3.840928521982809e-5 - 0.0007297009946044913*y + 0.0005512928309135257*x + 0.12016142738214425*y^2 + 0.09014586674317034*x*y + 0.06695613335029466*x^2)^2 + (-6.129681197559563e-7 - 0.03995870364206925*y + 0.05339661114720622*x - 0.0002606590185357672*y^2 - 0.00019560813848566583*x*y - 0.00014398491757863224*x^2)^2

We can verify that it is correct as follows:

In [7]:
rem(dec - poly, S.I)

6.35395673959307573778036997043383582450815083575434982776641845703125e-9 - 2.034188582850762212003499359917675245941090440446248230261680407401806641379595e-7y - 1.611004459406855788144569557446699876051682692307692307692307692307688893084131e-7x - 1.303706300006495018175937949944653837519581429660320281982421875e-5y² - 1.952346687528565735681405912682606640373705886304378509521484375e-5xy - 7.29787637974253704611395099366433214527205564081668853759765625e-6x² - 4.65613272382370269042439758777618408203125e-8y³ - 5.0109091542793748885742388665676116943359375e-8xy² - 4.185931611484558349280116207324908828013576567173004150390625e-8x²y + 1.289853256681593708120859586275540865384615384615384615384615384615111446730003e-8y⁴

Note that the difference between `dec` and `poly` is larger
than between the full gram matrix because `dec` is obtained by dropping
the lowest eigenvalues with the threshold `1e-6`; see `sos_decomposition`.

In [8]:
rem(gram_matrix(con_ref) - poly, S.I)

7.4087384965815732753675206911751771610141759083489887416362762451171875e-9 + 1.297590991011211291115404963372808147608175670775847557263496594551645124665685e-8y + 8.5589374903677778023008543711442213792067307692307692307692307692341838389502e-10x + 7.4087505692009558710964256533770821988582611083984375e-9y² + 1.82909243306994540034793317317962646484375e-14xy + 7.408704378285280167659720973460935056209564208984375e-9x² - 4.25141403563126611212889353434244791666666666666666666666666666712726232293837e-13y³ + 6.793084613339791152005394299825032552083333333333333333333333333563631161469185e-13xy² + 4.59667026664334343877271749079227447509765625e-13x²y + 1.295709003338393253775743337777944711538461538461538461538461538461265292883881e-8y⁴

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*